In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [40]:
# Install the lightning-fast C++ OSM extraction tool
!apt-get install -y osmium-tool osmium
!pip install geopandas osmnx

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package osmium


In [42]:
# Install the pre-compiled pyosmium library directly via apt-get
!apt-get update
!apt-get install -y python3-pyosmium

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-pyosmium is already the newest version (3.7.0-1build1).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.


In [14]:
!pip install pyosmium

ERROR: Could not find a version that satisfies the requirement pyosmium (from versions: none)
ERROR: No matching distribution found for pyosmium


STTEP 1

In [43]:
# 6. Visualize both routes and the flood polygon on an interactive Folium Map
import folium
import osmnx as ox
import networkx as nx
import geopandas as gpd

# Ensure routing variables exist
if 'G' not in globals() or 'baseline_route' not in globals():
    print("Routing variables missing. Setting up road graph and routes first...")
    place_name = "Silchar, Assam, India"
    G = ox.graph_from_place(place_name, network_type="drive")
    flood_gdf = gpd.read_file("/content/drive/MyDrive/AI Builder 2026/Silchar_Flood_Polygon_2022.geojson")

    G_proj = ox.project_graph(G)
    flood_proj = flood_gdf.to_crs(G_proj.graph["crs"])
    nodes, edges = ox.graph_to_gdfs(G_proj)
    flooded_edges = gpd.sjoin(edges, flood_proj, predicate='intersects')

    G_scenario = G_proj.copy()
    for u, v, key in flooded_edges.index:
        if G_scenario.has_edge(u, v, key):
            G_scenario.remove_edge(u, v, key)

    orig_lat, orig_lon = 24.8100, 92.7500
    dest_lat, dest_lon = 24.8010, 92.7980
    orig_node = ox.distance.nearest_nodes(G, orig_lon, orig_lat)
    dest_node = ox.distance.nearest_nodes(G, dest_lon, dest_lat)

    baseline_route = ox.routing.shortest_path(G_proj, orig_node, dest_node, weight="length")
    scenario_route = ox.routing.shortest_path(G_scenario, orig_node, dest_node, weight="length")

print("Generating interactive map...")

# Center map directly on the route endpoints
center_lat = (orig_lat + dest_lat) / 2
center_lon = (orig_lon + dest_lon) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery'
)

# Convert unprojected node paths to lat/lon tuples
def route_to_coords(graph, route):
    return [(graph.nodes[node]['y'], graph.nodes[node]['x']) for node in route]

baseline_coords = route_to_coords(G, baseline_route)
scenario_coords = route_to_coords(G, scenario_route)

# 1. Filter out SAR speckle noise and render clean translucent flood water
# Project to metric CRS to calculate real-world area, then back to WGS84
flood_metric = flood_gdf.to_crs(epsg=32646) if flood_gdf.crs != 32646 else flood_gdf
flood_cleaned = flood_metric[flood_metric.geometry.area > 5000].to_crs(epsg=4326)

folium.GeoJson(
    flood_cleaned,
    name="Sentinel-1 Flood Water",
    style_function=lambda x: {
        "fillColor": "#00BFFF",   # Vibrant deep sky blue
        "color": "#00BFFF",
        "weight": 0,              # KEY FIX: zero border weight eliminates the pixel artifact grid
        "fillOpacity": 0.55,
    }
).add_to(m)

# 2. Draw Baseline Route (Solid Emerald Green)
folium.PolyLine(
    baseline_coords,
    color="#00FF66",
    weight=5,
    opacity=0.9,
    tooltip="Baseline Route (Normal)"
).add_to(m)

# 3. Draw Scenario Route (Dashed Amber/Orange)
folium.PolyLine(
    scenario_coords,
    color="#FF3300",
    weight=4,
    opacity=0.9,
    tooltip="Disaster Detour Route",
    dash_array="6, 8"
).add_to(m)

# 4. Origin and Destination Markers
folium.Marker(
    location=[orig_lat, orig_lon],
    popup="Origin Settlement",
    icon=folium.Icon(color="green", icon="home")
).add_to(m)

folium.Marker(
    location=[dest_lat, dest_lon],
    popup="Destination Health Facility",
    icon=folium.Icon(color="red", icon="plus-sign")
).add_to(m)

# 5. Auto-fit camera bounds directly to the route extents
all_points = baseline_coords + scenario_coords
m.fit_bounds([[min(p[0] for p in all_points), min(p[1] for p in all_points)],
              [max(p[0] for p in all_points), max(p[1] for p in all_points)]])

folium.LayerControl().add_to(m)

# Display map
m

Generating interactive map...


In [23]:
%%bash
# Correct path to your uploaded PBF file in Google Drive
INPUT_PBF="/content/drive/MyDrive/AI Builder 2026/north-eastern-zone-261011.osm.pbf"
OUTPUT_PBF="/content/cachar_bbox.osm.pbf"

# Clip using bounding box: min_lon,min_lat,max_lon,max_lat
osmium extract -b 92.5,24.5,93.2,25.1 "$INPUT_PBF" -o "$OUTPUT_PBF" --overwrite

echo "Clipped PBF created successfully!"

Clipped PBF created successfully!


STEP 2

In [44]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import folium

# 1. Download Silchar's road network directly from OSM into memory
print("Downloading road graph...")
place_name = "Silchar, Assam, India"
G = ox.graph_from_place(place_name, network_type="drive")

# 2. Load your Sentinel-1 Flood Polygon from Google Drive
print("Loading flood polygon...")
flood_gdf = gpd.read_file("/content/drive/MyDrive/AI Builder 2026/Silchar_Flood_Polygon_2022.geojson")

# Project graph to UTM so we can calculate distances in meters
G_proj = ox.project_graph(G)
flood_proj = flood_gdf.to_crs(G_proj.graph["crs"])

# 3. Identify and sever the flooded roads
print("Severing flooded roads...")
nodes, edges = ox.graph_to_gdfs(G_proj)

# Find which road segments physically intersect the flood water
flooded_edges = gpd.sjoin(edges, flood_proj, predicate='intersects')

# Create a copy of the graph for the disaster scenario and remove the flooded edges
G_scenario = G_proj.copy()
for u, v, key in flooded_edges.index:
    if G_scenario.has_edge(u, v, key):
        G_scenario.remove_edge(u, v, key)

# 4. Define Start (a rural village) and End (Silchar Medical College)
orig_lat, orig_lon = 24.8100, 92.7500 # Example starting village
dest_lat, dest_lon = 24.8010, 92.7980 # Example hospital

# Snap coordinates using the unprojected graph G (which is in degrees)
orig_node = ox.distance.nearest_nodes(G, orig_lon, orig_lat)
dest_node = ox.distance.nearest_nodes(G, dest_lon, dest_lat)

# Helper function to get route length safely in newer OSMnx versions
def get_route_length(graph, route):
    if not route or len(route) < 2:
        return 0
    # Get the edges of the route and sum their lengths
    edge_lengths = ox.routing.route_to_gdf(graph, route)["length"]
    return edge_lengths.sum()

# 5. Run the Dijkstra algorithm for both scenarios
print("Calculating baseline route...")
try:
    baseline_route = ox.routing.shortest_path(G_proj, orig_node, dest_node, weight="length")
    baseline_length = get_route_length(G_proj, baseline_route)
    print(f"Normal distance: {baseline_length / 1000:.2f} km")
except nx.NetworkXNoPath:
    print("No baseline route exists.")

print("Calculating disaster route...")
try:
    scenario_route = ox.routing.shortest_path(G_scenario, orig_node, dest_node, weight="length")
    scenario_length = get_route_length(G_scenario, scenario_route)
    print(f"Disaster distance: {scenario_length / 1000:.2f} km")
except nx.NetworkXNoPath:
    print("VILLAGE IS COMPLETELY ISOLATED. No route exists.")

Loading flood polygon...
Severing flooded roads...
Calculating baseline route...
Normal distance: 6.91 km
Calculating disaster route...
Disaster distance: 7.17 km


STEP 3

In [22]:
import geopandas as gpd
import os
import subprocess

pbf_path = "/content/cachar_bbox.osm.pbf"
input_pbf_drive = "/content/drive/MyDrive/AI Builder 2026/north-eastern-zone-261011.osm.pbf"

# 1. Automatically perform the clip using osmium CLI if the output file does not exist
if not os.path.exists(pbf_path):
    print(f"Clipped PBF '{pbf_path}' not found. Initializing clipping process from Google Drive...")
    if os.path.exists(input_pbf_drive):
        subprocess.run("apt-get install -y osmium-tool", shell=True, check=True)
        clip_cmd = f'osmium extract -b 92.5,24.5,93.2,25.1 "{input_pbf_drive}" -o "{pbf_path}" --overwrite'
        subprocess.run(clip_cmd, shell=True, check=True)
        print("Clipped PBF created successfully!")
    else:
        print(f"Error: Input PBF file not found at '{input_pbf_drive}'")

# 2. Read points layer natively from the clipped local PBF file
print("Reading health facilities and villages natively from local clipped PBF file...")
try:
    # Read only the 'points' layer
    gdf_points = gpd.read_file(pbf_path, layer='points')

    # Function to parse tags from OGR's 'other_tags' key-value list (e.g. '"amenity"=>"hospital","place"=>"village"')
    def get_tag_value(other_tags_str, target_key):
        if not other_tags_str:
            return None
        # Simple parser for hstore-like formatting
        parts = other_tags_str.split(',')
        for part in parts:
            if '=>' in part:
                key, val = part.split('=>', 1)
                key = key.strip(' "')
                val = val.strip(' "')
                if key == target_key:
                    return val
        return None

    # Populate columns if missing
    if 'amenity' not in gdf_points.columns:
        gdf_points['amenity'] = gdf_points['other_tags'].apply(lambda x: get_tag_value(x, 'amenity'))
    if 'place' not in gdf_points.columns:
        gdf_points['place'] = gdf_points['other_tags'].apply(lambda x: get_tag_value(x, 'place'))
    if 'name' not in gdf_points.columns:
        gdf_points['name'] = gdf_points['other_tags'].apply(lambda x: get_tag_value(x, 'name'))

    # Filter health facilities
    health_mask = gdf_points['amenity'].isin(['hospital', 'clinic', 'doctors', 'health_post'])
    gdf_health = gdf_points[health_mask].copy()
    gdf_health['name'] = gdf_health['name'].fillna('Unnamed Health Facility')
    gdf_health = gdf_health[['name', 'amenity', 'geometry']].dropna(subset=['geometry'])

    # Filter villages/settlements
    village_mask = gdf_points['place'].isin(['village', 'hamlet', 'town', 'isolated_dwelling'])
    gdf_villages = gdf_points[village_mask].copy()
    gdf_villages['name'] = gdf_villages['name'].fillna('Unnamed Settlement')
    gdf_villages = gdf_villages[['name', 'place', 'geometry']].dropna(subset=['geometry'])

except Exception as e:
    print(f"PBF reading failed: {e}")
    gdf_health = gpd.GeoDataFrame(columns=['name', 'amenity', 'geometry'], crs="EPSG:4326")
    gdf_villages = gpd.GeoDataFrame(columns=['name', 'place', 'geometry'], crs="EPSG:4326")

print(f"Extracted {len(gdf_health)} verified health facilities.")
print(f"Extracted {len(gdf_villages)} villages/settlements.")

Reading health facilities and villages natively from local clipped PBF file...
Extracted 55 verified health facilities.
Extracted 334 villages/settlements.


STEP 4

In [24]:
import pandas as pd
import networkx as nx
import osmnx as ox

print("Optimizing calculation using multi-source Dijkstra...")

# 1. Project geometries and convert centroids to x, y arrays
gdf_villages_proj = gdf_villages.to_crs(G_proj.graph["crs"])
gdf_health_proj = gdf_health.to_crs(G_proj.graph["crs"])

v_centroids = gdf_villages.geometry.centroid
h_centroids = gdf_health.geometry.centroid

# 2. Vectorized snapping of all points to graph nodes in one step
print("Snapping nodes to the network...")
villages_nodes = ox.distance.nearest_nodes(G, v_centroids.x.values, v_centroids.y.values)
health_nodes = ox.distance.nearest_nodes(G, h_centroids.x.values, h_centroids.y.values)

# Create mappings
village_to_node = dict(zip(gdf_villages.index, villages_nodes))
health_to_node = dict(zip(gdf_health.index, health_nodes))
node_to_health_name = {health_to_node[idx]: gdf_health.loc[idx, 'name'] for idx in gdf_health.index}

# Keep track of unique targets
health_node_set = set(health_nodes)

results = []

# 3. Use NetworkX's highly optimized single-source Dijkstra to calculate distances
# from each village to ALL reachable health nodes in one pass per village
print("Computing shortest paths...")
for idx, v_row in gdf_villages.iterrows():
    orig_node = village_to_node[idx]
    v_name = v_row['name']
    v_pt = v_centroids.loc[idx]

    # Baseline distances from this village node to all reachable nodes in G_proj
    try:
        baseline_paths = nx.single_source_dijkstra_path_length(G_proj, orig_node, weight="length")
        # Filter only to destinations that are health facilities
        reachable_health_baseline = {node: dist for node, dist in baseline_paths.items() if node in health_node_set}
    except Exception:
        reachable_health_baseline = {}

    if not reachable_health_baseline:
        continue  # Isolated baseline

    # Find the nearest health facility node
    best_h_node = min(reachable_health_baseline, key=reachable_health_baseline.get)
    nearest_h_dist = reachable_health_baseline[best_h_node]
    best_h_name = node_to_health_name[best_h_node]

    # Scenario distances from this village node to all reachable nodes in G_scenario
    try:
        scenario_paths = nx.single_source_dijkstra_path_length(G_scenario, orig_node, weight="length")
        scen_dist = scenario_paths.get(best_h_node, None)
    except Exception:
        scen_dist = None

    # Determine flood impact
    if scen_dist is not None:
        if nearest_h_dist > 0:
            detour_ratio = scen_dist / nearest_h_dist
            status = "Detoured" if detour_ratio > 1.2 else "Unaffected"
        else:
            detour_ratio = 1.0
            status = "Unaffected"
    else:
        detour_ratio = None
        status = "ISOLATED"

    results.append({
        'village': v_name,
        'coords': [v_pt.y, v_pt.x],
        'target_facility': best_h_name,
        'baseline_km': round(nearest_h_dist / 1000, 2),
        'disaster_km': round(scen_dist / 1000, 2) if scen_dist is not None else None,
        'status': status
    })

# Convert to DataFrame and sort
df_results = pd.DataFrame(results)
isolated_villages = df_results[df_results['status'] == 'ISOLATED']
detoured_villages = df_results[df_results['status'] == 'Detoured']

print(f"\nTotal Villages Evaluated: {len(df_results)}")
print(f"Isolated Villages: {len(isolated_villages)}")
print(f"Detoured Villages: {len(detoured_villages)}")
display(df_results.head(10))

Optimizing calculation using multi-source Dijkstra...


/tmp/ipykernel_1889/3055315021.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  v_centroids = gdf_villages.geometry.centroid
/tmp/ipykernel_1889/3055315021.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  h_centroids = gdf_health.geometry.centroid


Snapping nodes to the network...
Computing shortest paths...

Total Villages Evaluated: 334
Isolated Villages: 15
Detoured Villages: 6


,village,coords,target_facility,baseline_km,disaster_km,status
0,Umednagar,"[24.567090800000003, 92.6165021]","State Dispensary,Nityanandpur",0.00,0.00,Unaffected
1,Pātbāri,"[24.591539100000002, 92.95313200000001]","Mini PHC, Narsingpur",11.20,11.20,Unaffected
2,Hailakandi,"[24.6846916, 92.5645836]",Pain Control Clinic,0.00,0.00,Unaffected
3,Sonāimukh,"[24.725556, 92.88115930000001]","BPHC,Sonai",0.00,0.00,Unaffected
4,Silkuri,"[24.7271083, 92.7840477]",R . E. Nursing Home and Diagnostic,6.94,6.94,Unaffected
5,Lakhipur,"[24.792822700000002, 93.0090947]",Jiribam Hospital,0.00,0.00,Unaffected
6,Udharbond,"[24.881959100000003, 92.88177540000001]",Joypur Government Hospital,2.16,2.16,Unaffected
7,Kumbhir,"[24.939853600000003, 92.96533000000001]",Thaligram Hospital,0.00,0.00,Unaffected
8,Katakhal,"[24.832364000000002, 92.6318201]",KATIGORAH MODEL HOSPITAL,0.00,0.00,Unaffected
9,Jiribam,"[24.8098254, 93.1137269]",Jiribam Hospital,0.00,0.00,Unaffected


In [25]:
# Sort the detoured villages by the severity of the detour (longest disaster route first)
if not detoured_villages.empty:
    detoured_villages['detour_severity'] = detoured_villages['disaster_km'] - detoured_villages['baseline_km']
    detoured_sorted = detoured_villages.sort_values(by='detour_severity', ascending=False)
    print("--- TOP SEVERELY DETOURED VILLAGES ---")
    display(detoured_sorted[['village', 'target_facility', 'baseline_km', 'disaster_km', 'status']].head(5))
else:
    print("No detoured villages found.")

print("\n--- ISOLATED VILLAGES ---")
if not isolated_villages.empty:
    display(isolated_villages[['village', 'target_facility', 'baseline_km', 'status']].head(5))
else:
    print("No isolated villages found.")

--- TOP SEVERELY DETOURED VILLAGES ---


/tmp/ipykernel_1889/3583763946.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  detoured_villages['detour_severity'] = detoured_villages['disaster_km'] - detoured_villages['baseline_km']


,village,target_facility,baseline_km,disaster_km,status
222,Boālipār,"Physiotherapy Clinic, Hailakandi",5.55,7.22,Detoured
323,Khelma,"Lions Eye Hospital, Cachar",3.73,5.09,Detoured
256,Māsimpur,Unnamed Health Facility,5.28,6.46,Detoured
322,Rāmnagar,"Lions Eye Hospital, Cachar",3.07,4.13,Detoured
321,Khelma,"Lions Eye Hospital, Cachar",4.69,5.72,Detoured



--- ISOLATED VILLAGES ---


,village,target_facility,baseline_km,status
16,Moragang,"State Dispensary,Nityanandpur",1.02,ISOLATED
23,Dwarband,"State Dispensary,Nityanandpur",6.28,ISOLATED
29,Diglāng,Mawsumi Hospital,7.26,ISOLATED
72,Diglāng,Mawsumi Hospital,7.26,ISOLATED
75,Nutan Bāzār,Mawsumi Hospital,7.26,ISOLATED


DIAGNOSIS

In [26]:

# Inspect suspicious rows before changing the routing logic
suspect_rows = df_results[
    df_results["target_facility"].isna()
    | df_results["baseline_km"].isna()
    | df_results["disaster_km"].isna()
    | (df_results["baseline_km"] <= 0)
    | (df_results["disaster_km"] <= 0)
]

display(suspect_rows)


,village,coords,target_facility,baseline_km,disaster_km,status
0,Umednagar,"[24.567090800000003, 92.6165021]","State Dispensary,Nityanandpur",0.0,0.0,Unaffected
2,Hailakandi,"[24.6846916, 92.5645836]",Pain Control Clinic,0.0,0.0,Unaffected
3,Sonāimukh,"[24.725556, 92.88115930000001]","BPHC,Sonai",0.0,0.0,Unaffected
5,Lakhipur,"[24.792822700000002, 93.0090947]",Jiribam Hospital,0.0,0.0,Unaffected
7,Kumbhir,"[24.939853600000003, 92.96533000000001]",Thaligram Hospital,0.0,0.0,Unaffected
...,...,...,...,...,...,...
328,Mauchār,"[25.021873900000003, 93.1974024]",Thaligram Hospital,0.0,0.0,Unaffected
329,Bara Ārkāp,"[25.032018200000003, 93.1930947]",Thaligram Hospital,0.0,0.0,Unaffected
331,Alni,"[24.732959500000003, 93.0440078]",Jiribam Hospital,0.0,0.0,Unaffected
332,Chandrapur,"[24.811008700000002, 92.92529950000001]",Jiribam Hospital,0.0,0.0,Unaffected


In [28]:

import pandas as pd

# 1. Count zero, missing, and valid distances
for col in ["baseline_km", "disaster_km"]:
    distances = pd.to_numeric(df_results[col], errors="coerce")

    print(f"\n{col}")
    print("Zero:", (distances == 0).sum())
    print("Negative:", (distances < 0).sum())
    print("Missing/invalid:", distances.isna().sum())
    print("Positive:", (distances > 0).sum())

# 2. Inspect rows with zero or missing route distances
mask = (
    df_results["baseline_km"].isna()
    | df_results["disaster_km"].isna()
    | (df_results["baseline_km"] <= 0)
    | (df_results["disaster_km"] <= 0)
)

display(
    df_results.loc[
        mask,
        ["village", "coords", "target_facility",
         "baseline_km", "disaster_km", "status"]
    ].head(30)
)



baseline_km
Zero: 108
Negative: 0
Missing/invalid: 0
Positive: 226

disaster_km
Zero: 108
Negative: 0
Missing/invalid: 15
Positive: 211


,village,coords,target_facility,baseline_km,disaster_km,status
0,Umednagar,"[24.567090800000003, 92.6165021]","State Dispensary,Nityanandpur",0.00,0.0,Unaffected
2,Hailakandi,"[24.6846916, 92.5645836]",Pain Control Clinic,0.00,0.0,Unaffected
3,Sonāimukh,"[24.725556, 92.88115930000001]","BPHC,Sonai",0.00,0.0,Unaffected
5,Lakhipur,"[24.792822700000002, 93.0090947]",Jiribam Hospital,0.00,0.0,Unaffected
7,Kumbhir,"[24.939853600000003, 92.96533000000001]",Thaligram Hospital,0.00,0.0,Unaffected
8,Katakhal,"[24.832364000000002, 92.6318201]",KATIGORAH MODEL HOSPITAL,0.00,0.0,Unaffected
9,Jiribam,"[24.8098254, 93.1137269]",Jiribam Hospital,0.00,0.0,Unaffected
14,Badarpur,"[24.8711319, 92.558515]",KATIGORAH MODEL HOSPITAL,0.00,0.0,Unaffected
15,Pailapool,"[24.8285252, 93.0253248]",Jiribam Hospital,0.00,0.0,Unaffected
16,Moragang,"[24.6312697, 92.6346292]","State Dispensary,Nityanandpur",1.02,NaN,ISOLATED


In [46]:
%%bash
echo "Extracting Healthcare Facilities..."
osmium tags-filter /content/cachar_bbox.osm.pbf n/amenity=hospital,clinic,doctors,health_post,dispensary -o health.osm.pbf --overwrite
osmium export health.osm.pbf -o health.geojson --geometry-types=point --overwrite

echo "Extracting Settlements..."
osmium tags-filter /content/cachar_bbox.osm.pbf n/place=village,hamlet,town,isolated_dwelling -o villages.osm.pbf --overwrite
osmium export villages.osm.pbf -o villages.geojson --geometry-types=point --overwrite

echo "GeoJSON extraction complete!"

Extracting Healthcare Facilities...
Extracting Settlements...
GeoJSON extraction complete!


Code fixes

In [ ]:
# ==========================================
# 5. MULTI-FACILITY DYNAMIC ACCESSIBILITY SCAN (OPTIMIZED)
# ==========================================
print("\nStep 5: Executing lightning-fast accessibility analysis...")

analysis_records = []

for v_idx, v_row in valid_villages.iterrows():
    v_node = v_row["graph_node"]
    v_name = v_row.get("name", "Unnamed Village")

    # 1. Baseline: Compute distance to ALL nodes in the network simultaneously
    try:
        base_all_dists = nx.single_source_dijkstra_path_length(G_proj, v_node, weight="length")
    except nx.NodeNotFound:
        continue

    # Filter down to just our healthcare facilities
    base_dists = {h: base_all_dists[h] / 1000.0 for h in all_health_nodes if h in base_all_dists}

    if not base_dists:
        continue

    best_base_h = min(base_dists, key=base_dists.get)
    base_km = base_dists[best_base_h]
    is_coincident = base_km < 0.05

    # 2. Scenario: Compute distance to ALL nodes in the flooded network simultaneously
    try:
        scen_all_dists = nx.single_source_dijkstra_path_length(G_scenario, v_node, weight="length")
    except nx.NodeNotFound:
        scen_all_dists = {}

    scen_dists = {h: scen_all_dists[h] / 1000.0 for h in operational_health_nodes if h in scen_all_dists}

    # Evaluate outcomes and handle missing routes properly
    if not scen_dists:
        scen_km = np.nan
        delta_km = np.nan
        detour_ratio = np.nan
        dest_changed = False
        status = "MODEL_ISOLATED"
        best_scen_h = None  # Key fix: explicit None for isolated villages
    else:
        best_scen_h = min(scen_dists, key=scen_dists.get)
        scen_km = scen_dists[best_scen_h]
        delta_km = round(scen_km - base_km, 2)
        detour_ratio = round(scen_km / base_km, 2) if base_km > 0.05 else np.nan
        dest_changed = best_scen_h != best_base_h

        if dest_changed:
            status = "REROUTED_ALT_FACILITY"
        else:
            status = "REACHABLE_SAME_FACILITY"

    analysis_records.append({
        "village_id": v_idx,
        "village_name": v_name,
        "lat": G.nodes[v_node]['y'],
        "lon": G.nodes[v_node]['x'],
        "snap_distance_m": round(v_row["snap_dist_m"], 1),
        "baseline_nearest_node": best_base_h,
        "scenario_nearest_node": best_scen_h,  # Now successfully tracked
        "baseline_km": round(base_km, 2),
        "scenario_km": round(scen_km, 2) if not np.isnan(scen_km) else None,
        "additional_km": delta_km,
        "detour_ratio": detour_ratio,
        "destination_altered": dest_changed,
        "is_coincident_base": is_coincident,
        "modeled_status": status,
    })

df_audit = pd.DataFrame(analysis_records)
df_clean = df_audit[~df_audit["is_coincident_base"]].copy()

print(f"\nAudit complete. Total valid analyzed settlements: {len(df_clean)}")
print(f"Model-Isolated settlements: {len(df_clean[df_clean['modeled_status'] == 'MODEL_ISOLATED'])}")
print(f"Rerouted to alternative facilities: {len(df_clean[df_clean['modeled_status'] == 'REROUTED_ALT_FACILITY'])}")
display(df_clean[["village_name", "baseline_km", "scenario_km", "additional_km", "detour_ratio", "modeled_status"]].sort_values(by="additional_km", ascending=False).head(10))


Step 5: Executing lightning-fast accessibility analysis...


In [58]:
# ==========================================
# 6. TARGETED INTERVENTION & VISUALIZER
# ==========================================
def generate_audit_map(target_df_index, restore_edges=False, targeted_edges=None):
    # 1. Fix Index Desync: Get the true original village ID
    row = df_clean.loc[target_df_index]
    true_v_idx = row["village_id"]

    v_node = int(valid_villages.loc[true_v_idx, "graph_node"])
    base_h_node = int(row["baseline_nearest_node"])
    scen_h_node = int(row["scenario_nearest_node"]) if pd.notna(row["scenario_nearest_node"]) else None

    # 2. Targeted Intervention Logic
    G_active = G_scenario.copy()
    if restore_edges and targeted_edges:
        for u, v, k in targeted_edges:
            if G_proj.has_edge(u, v, k):
                data = G_proj.get_edge_data(u, v, k)
                G_active.add_edge(u, v, k, **data)
            if G_proj.has_edge(v, u, k):
                data = G_proj.get_edge_data(v, u, k)
                G_active.add_edge(v, u, k, **data)

    # 3. Calculate True Routes
    base_path = ox.routing.shortest_path(G_proj, v_node, base_h_node, weight="length")

    active_path = None
    if scen_h_node is not None:
        try:
            # Route to the SCENARIO facility, not the baseline one!
            active_path = ox.routing.shortest_path(G_active, v_node, scen_h_node, weight="length")
        except nx.NetworkXNoPath:
            pass

    # 4. Map Rendering
    m = folium.Map(
        location=[row["lat"], row["lon"]],
        zoom_start=13,
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery"
    )

    flood_metric = flood_gdf.to_crs(epsg=32646) if flood_gdf.crs != 32646 else flood_gdf
    flood_cleaned = flood_metric[flood_metric.geometry.area > 5000].to_crs(epsg=4326)
    folium.GeoJson(
        flood_cleaned,
        name="Modeled Inundation Footprint",
        style_function=lambda x: {"fillColor": "#00BFFF", "color": "#00BFFF", "weight": 0, "fillOpacity": 0.5}
    ).add_to(m)

    # Plot Baseline
    if base_path:
        base_coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in base_path]
        folium.PolyLine(base_coords, color="#00FF66", weight=5, opacity=0.8, tooltip="Baseline Route").add_to(m)
        folium.Marker(
            [G.nodes[base_h_node]['y'], G.nodes[base_h_node]['x']],
            popup="Baseline Health Facility",
            icon=folium.Icon(color="lightgreen", icon="plus-sign")
        ).add_to(m)

    # Plot Scenario Detour
    if active_path:
        active_coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in active_path]
        folium.PolyLine(
            active_coords,
            color="#FF3300" if not restore_edges else "#FFCC00",
            weight=4, dash_array="6, 8",
            tooltip="Active Detour Route" if not restore_edges else "Post-Intervention Route"
        ).add_to(m)

        # If rerouted, show the new destination facility
        if scen_h_node != base_h_node:
            folium.Marker(
                [G.nodes[scen_h_node]['y'], G.nodes[scen_h_node]['x']],
                popup="Alternative Facility (Rerouted)",
                icon=folium.Icon(color="red", icon="plus-sign")
            ).add_to(m)

    # Origin Village
    folium.Marker(
        [row["lat"], row["lon"]],
        popup=f"Settlement: {row['village_name']}",
        icon=folium.Icon(color="green", icon="home")
    ).add_to(m)

    return m

# 1. Reveal the 8 completely isolated settlements
isolated_df = df_clean[df_clean['modeled_status'] == 'MODEL_ISOLATED']
print("--- CRITICAL: COMPLETELY ISOLATED SETTLEMENTS ---")
display(isolated_df[['village_id', 'village_name', 'baseline_km', 'modeled_status']])

# 2. Visually audit the massive detour for Binnākāndi (Index 64)
print("\nGenerating visual audit for Binnākāndi...")
map_binnakandi = generate_audit_map(64, restore_edges=False)
map_binnakandi

--- CRITICAL: COMPLETELY ISOLATED SETTLEMENTS ---


,village_id,village_name,baseline_km,modeled_status
14,14,Badarpur,4.59,MODEL_ISOLATED
68,72,Diglāng,8.50,MODEL_ISOLATED
82,88,Duiheng,54.64,MODEL_ISOLATED
83,89,Ārda,46.08,MODEL_ISOLATED
87,94,Disāma,40.46,MODEL_ISOLATED
88,95,Disāma,40.46,MODEL_ISOLATED
219,228,Rāmnagar,5.30,MODEL_ISOLATED
291,301,Digli,8.50,MODEL_ISOLATED



Generating visual audit for Binnākāndi...


KeyError: 'scenario_nearest_node'